# **🩺 Clean & Jerk Dataset — Fault Inspection & Logger**

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import (
    DEFAULT_CLEAN_DATASET_DIR,
    detect_set_faults,
    launch_fault_logger,
    load_set_h5,
    plot_joint_4panel,
)

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
print(f"📂 Dataset folder: {dataset_dir}")

### 1️⃣ Load Set & Auto-Detected Faults Summary

In [ ]:
# @title 1️⃣ Load Set & Auto-Detected Faults { display-mode: "form" }
set_id = "set_0001" # @param {type:"string"}
set_data = load_set_h5(dataset_dir, set_id)

meta = set_data["metadata"]
print(
    f"✅ {set_id} | Participant: {meta.get('participant_id')} | "
    f"Lifts: {meta.get('lifts_total')} | Duration: {meta.get('duration_s')}s | "
    f"Sensors OK: {meta.get('sensors_ok')}/{meta.get('sensors_total')} | "
    f"has_faults: {meta.get('has_faults')}"
)

df_auto_faults = detect_set_faults(set_data, freeze_sec=0.5)
print(f"🔎 Auto-detected channel issues: {len(df_auto_faults)}")
display_table(df_auto_faults, clean_columns=False)

### 2️⃣ Joint-Level 4-Subplot IMU & Gyro Inspector

In [ ]:
# @title 2️⃣ Joint 4-Subplot Inspector (Full Set or Individual Lift) { display-mode: "form" }
joint_name = "right_knee" # @param ["right_knee","left_knee","right_hip","left_hip","right_ankle","left_ankle","right_shoulder","left_shoulder","right_elbow","left_elbow","right_wrist","left_wrist","spine"]
lift_id = 0 # @param {type:"integer"}  # 0 = Full Set, 1..N = Individual Lift

fig_joint = plot_joint_4panel(
    set_data,
    joint_name=joint_name,
    lift_id=None if lift_id == 0 else lift_id,
)
fig_joint.show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})

### 3️⃣ Log / Edit Faults (`fault_log.csv`)

In [ ]:
# @title 3️⃣ Interactive Fault Logger
launch_fault_logger(dataset_dir, set_data)